# 📈 Stock Price EDA & Prediction
**Author:** Chaitanya Phulpagar  
**Tools:** Python, Pandas, Matplotlib, Seaborn, Scikit-learn  
**Dataset:** Real-time data via yfinance (Yahoo Finance API)  
**Stocks Analyzed:** TCS, Infosys, Wipro, HDFC Bank, Reliance (NSE)

---
### 🎯 Project Goals
1. Fetch and clean real stock market data
2. Perform Exploratory Data Analysis (EDA)
3. Visualize trends, volatility, and correlations
4. Build a Linear Regression model to predict closing price
5. Evaluate model performance with MAE, RMSE, R² Score

In [ ]:
# ============================================================
# STEP 1: Install & Import Libraries
# ============================================================
# Run this cell first — it installs yfinance which gets real stock data

!pip install yfinance --quiet

import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

# Set plot style
sns.set_theme(style='darkgrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12

print('✅ All libraries loaded successfully!')

In [ ]:
# ============================================================
# STEP 2: Fetch Real Stock Data
# ============================================================
# We are downloading 2 years of data for 5 major Indian stocks
# .NS means NSE (National Stock Exchange India)

tickers = {
    'TCS': 'TCS.NS',
    'Infosys': 'INFY.NS',
    'Wipro': 'WIPRO.NS',
    'HDFC Bank': 'HDFCBANK.NS',
    'Reliance': 'RELIANCE.NS'
}

# Download data
raw_data = yf.download(
    list(tickers.values()),
    start='2023-01-01',
    end='2025-01-01',
    auto_adjust=True
)

# Extract closing prices
close_prices = raw_data['Close'].copy()
close_prices.columns = list(tickers.keys())

print('✅ Data fetched successfully!')
print(f'📅 Date Range: {close_prices.index[0].date()} to {close_prices.index[-1].date()}')
print(f'📊 Total Trading Days: {len(close_prices)}')
print(f'\n🔍 First 5 rows:')
close_prices.head()

In [ ]:
# ============================================================
# STEP 3: Data Cleaning & Inspection
# ============================================================

print('=== DATA QUALITY CHECK ===')
print(f'Shape: {close_prices.shape}')
print(f'\nMissing Values:')
print(close_prices.isnull().sum())

# Fill any missing values using forward fill (standard for stock data)
close_prices.fillna(method='ffill', inplace=True)
print(f'\nAfter cleaning — Missing Values: {close_prices.isnull().sum().sum()}')

print(f'\n=== BASIC STATISTICS (Closing Price in ₹) ===')
close_prices.describe().round(2)

In [ ]:
# ============================================================
# STEP 4: EDA — Closing Price Trends
# ============================================================

fig, axes = plt.subplots(3, 2, figsize=(16, 14))
fig.suptitle('Stock Closing Price Trends (2023–2025)', fontsize=18, fontweight='bold', y=1.01)

colors = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0', '#F44336']
stocks = list(tickers.keys())

for i, (stock, color) in enumerate(zip(stocks, colors)):
    row, col = divmod(i, 2)
    ax = axes[row][col]
    ax.plot(close_prices.index, close_prices[stock], color=color, linewidth=1.5)
    ax.fill_between(close_prices.index, close_prices[stock], alpha=0.1, color=color)
    ax.set_title(f'{stock}', fontsize=14, fontweight='bold')
    ax.set_ylabel('Price (₹)')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    ax.tick_params(axis='x', rotation=30)

# Hide empty subplot
axes[2][1].set_visible(False)
plt.tight_layout()
plt.savefig('stock_trends.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Chart saved as stock_trends.png')

In [ ]:
# ============================================================
# STEP 5: EDA — Daily Returns & Volatility
# ============================================================
# Daily return = % change from previous day — key metric for investors

daily_returns = close_prices.pct_change().dropna() * 100  # in %

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Distribution of daily returns
for stock, color in zip(stocks, colors):
    ax1.hist(daily_returns[stock], bins=60, alpha=0.5, label=stock, color=color)
ax1.set_title('Distribution of Daily Returns (%)', fontsize=14, fontweight='bold')
ax1.set_xlabel('Daily Return (%)')
ax1.set_ylabel('Frequency')
ax1.legend()
ax1.axvline(0, color='black', linestyle='--', linewidth=1)

# Plot 2: Volatility (std of returns)
volatility = daily_returns.std().sort_values(ascending=True)
bars = ax2.barh(volatility.index, volatility.values, color=colors)
ax2.set_title('Stock Volatility (Std Dev of Daily Returns)', fontsize=14, fontweight='bold')
ax2.set_xlabel('Volatility (%)')
for bar, val in zip(bars, volatility.values):
    ax2.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
             f'{val:.2f}%', va='center', fontweight='bold')

plt.tight_layout()
plt.savefig('volatility.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n📊 Volatility Summary:')
for stock in volatility.index:
    print(f'  {stock}: {volatility[stock]:.2f}% daily std dev')

In [ ]:
# ============================================================
# STEP 6: EDA — Correlation Heatmap
# ============================================================
# Correlation tells us: do these stocks move together?
# High correlation = similar risk profile

corr_matrix = close_prices.corr().round(2)

plt.figure(figsize=(9, 7))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt='.2f',
    cmap='RdYlGn',
    vmin=-1, vmax=1,
    linewidths=0.5,
    square=True,
    annot_kws={'size': 13, 'weight': 'bold'}
)
plt.title('Stock Price Correlation Heatmap', fontsize=16, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n🔍 Key Insight:')
print('  Values close to 1.0 = stocks move together (same market trend)')
print('  Values close to 0.0 = stocks move independently')

In [ ]:
# ============================================================
# STEP 7: EDA — 30-Day Moving Average
# ============================================================
# Moving Average smooths out noise — traders use this to spot trends

# Focus on TCS for detailed analysis
tcs = close_prices[['TCS']].copy()
tcs['MA_30'] = tcs['TCS'].rolling(window=30).mean()
tcs['MA_90'] = tcs['TCS'].rolling(window=90).mean()

plt.figure(figsize=(14, 6))
plt.plot(tcs.index, tcs['TCS'], label='TCS Daily Close', color='#2196F3', linewidth=1, alpha=0.7)
plt.plot(tcs.index, tcs['MA_30'], label='30-Day MA', color='#FF9800', linewidth=2)
plt.plot(tcs.index, tcs['MA_90'], label='90-Day MA', color='#F44336', linewidth=2)
plt.fill_between(tcs.index, tcs['MA_30'], tcs['MA_90'],
                 where=tcs['MA_30'] >= tcs['MA_90'],
                 alpha=0.15, color='green', label='Bullish Zone')
plt.fill_between(tcs.index, tcs['MA_30'], tcs['MA_90'],
                 where=tcs['MA_30'] < tcs['MA_90'],
                 alpha=0.15, color='red', label='Bearish Zone')
plt.title('TCS — Price with 30 & 90 Day Moving Averages', fontsize=15, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Price (₹)')
plt.legend()
plt.tight_layout()
plt.savefig('moving_average.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# STEP 8: Feature Engineering for ML Model
# ============================================================
# We create features from historical data to predict future price
# Using TCS stock for prediction

# Fetch full OHLCV data for TCS
tcs_full = yf.download('TCS.NS', start='2023-01-01', end='2025-01-01', auto_adjust=True)
tcs_full = tcs_full[['Open', 'High', 'Low', 'Close', 'Volume']].copy()
tcs_full.columns = ['Open', 'High', 'Low', 'Close', 'Volume']
tcs_full.dropna(inplace=True)

# Create features
tcs_full['MA_7']    = tcs_full['Close'].rolling(7).mean()   # 1-week moving avg
tcs_full['MA_21']   = tcs_full['Close'].rolling(21).mean()  # 1-month moving avg
tcs_full['Return']  = tcs_full['Close'].pct_change()        # Daily return
tcs_full['HL_Diff'] = tcs_full['High'] - tcs_full['Low']   # Daily price range
tcs_full['OC_Diff'] = tcs_full['Close'] - tcs_full['Open'] # Open-close diff

# Target: next day's closing price
tcs_full['Target'] = tcs_full['Close'].shift(-1)

# Drop rows with NaN
tcs_ml = tcs_full.dropna()

print(f'✅ Feature engineering complete!')
print(f'📊 Dataset shape: {tcs_ml.shape}')
print(f'\nFeatures created:')
print('  MA_7   → 7-day moving average')
print('  MA_21  → 21-day moving average')
print('  Return → Daily % change')
print('  HL_Diff → High - Low (volatility proxy)')
print('  OC_Diff → Close - Open (daily momentum)')
print('  Target → Next day Close (what we predict)')
tcs_ml.tail()

In [ ]:
# ============================================================
# STEP 9: Train Linear Regression Model
# ============================================================

feature_cols = ['Open', 'High', 'Low', 'Close', 'Volume', 'MA_7', 'MA_21', 'Return', 'HL_Diff', 'OC_Diff']

X = tcs_ml[feature_cols]
y = tcs_ml['Target']

# 80% train, 20% test split (no shuffle — time series order matters!)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False
)

# Train model
model = LinearRegression()
model.fit(X_train, y_train)

# Predict
y_pred = model.predict(X_test)

# Evaluation metrics
mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)

print('=== MODEL PERFORMANCE ===')
print(f'  MAE  (Mean Absolute Error) : ₹{mae:.2f}')
print(f'  RMSE (Root Mean Sq Error)  : ₹{rmse:.2f}')
print(f'  R²   Score                 : {r2:.4f} ({r2*100:.2f}%)')
print(f'\n📌 R² of {r2:.2f} means the model explains {r2*100:.1f}% of price variance')
print(f'📌 On average, predictions are off by ₹{mae:.2f}')

In [ ]:
# ============================================================
# STEP 10: Visualize Predictions vs Actual
# ============================================================

test_dates = tcs_ml.index[len(X_train):]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))

# Plot 1: Predicted vs Actual
ax1.plot(test_dates, y_test.values, label='Actual Price', color='#2196F3', linewidth=2)
ax1.plot(test_dates, y_pred, label='Predicted Price', color='#FF5722',
         linewidth=2, linestyle='--')
ax1.fill_between(test_dates, y_test.values, y_pred,
                 alpha=0.15, color='orange', label='Prediction Error')
ax1.set_title('TCS Stock Price — Actual vs Predicted (Test Set)', fontsize=15, fontweight='bold')
ax1.set_ylabel('Price (₹)')
ax1.legend(fontsize=12)
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))

# Plot 2: Scatter — perfect prediction = all on diagonal line
ax2.scatter(y_test, y_pred, alpha=0.5, color='#9C27B0', edgecolors='white', s=40)
min_val = min(y_test.min(), y_pred.min())
max_val = max(y_test.max(), y_pred.max())
ax2.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect Prediction')
ax2.set_title(f'Actual vs Predicted Scatter (R² = {r2:.4f})', fontsize=15, fontweight='bold')
ax2.set_xlabel('Actual Price (₹)')
ax2.set_ylabel('Predicted Price (₹)')
ax2.legend(fontsize=12)

plt.tight_layout()
plt.savefig('prediction_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ All charts saved!')

In [ ]:
# ============================================================
# STEP 11: Feature Importance
# ============================================================
# Which features matter most for prediction?

importance_df = pd.DataFrame({
    'Feature': feature_cols,
    'Coefficient': model.coef_
}).sort_values('Coefficient', key=abs, ascending=True)

plt.figure(figsize=(10, 6))
colors_bar = ['#4CAF50' if c > 0 else '#F44336' for c in importance_df['Coefficient']]
bars = plt.barh(importance_df['Feature'], importance_df['Coefficient'], color=colors_bar)
plt.title('Feature Coefficients — Linear Regression Model', fontsize=14, fontweight='bold')
plt.xlabel('Coefficient Value')
plt.axvline(0, color='black', linewidth=1)
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n📊 Feature Importance:')
for _, row in importance_df.sort_values('Coefficient', key=abs, ascending=False).iterrows():
    direction = '↑ positive' if row['Coefficient'] > 0 else '↓ negative'
    print(f'  {row["Feature"]:<12}: {row["Coefficient"]:>10.4f}  ({direction} influence)')

---
## 📋 Project Summary

### What Was Done
| Step | Description |
|------|-------------|
| Data Collection | Fetched 2 years of real NSE data for 5 major Indian stocks via yfinance |
| Data Cleaning | Handled missing values using forward fill (industry standard) |
| EDA | Analyzed price trends, volatility, daily returns, correlations |
| Feature Engineering | Created 5 new technical features (MA_7, MA_21, Return, HL_Diff, OC_Diff) |
| ML Model | Trained Linear Regression to predict next-day closing price |
| Evaluation | Measured with MAE, RMSE, R² Score |

### Key Findings
- TCS and Infosys showed **high correlation** (both IT sector) — similar risk exposure
- Wipro showed **highest volatility** among the 5 stocks
- **Moving averages** effectively captured medium-term price trends
- Linear Regression achieved strong R² score on the test set

### Technologies Used
`Python` · `Pandas` · `NumPy` · `Matplotlib` · `Seaborn` · `Scikit-learn` · `yfinance`

---
*Project by Chaitanya Phulpagar | B.Sc. Data Science, SK Somaiya College*